# Data Cleaning — U.S. Inflation and Resource Prices

I cleaned the two Kaggle datasets before analyzing them. I converted the dates, calculated percentage changes, and combined both datasets by month. The original CSV files are not changed.

## 1. Import libraries and load the raw data

First, I loaded both CSV files and checked the columns. I kept the raw files unchanged.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"

OUT_DIR.mkdir(parents=True, exist_ok=True)

cpi_raw = pd.read_csv(RAW_DIR / "US CPI.csv")
resources_raw = pd.read_csv(RAW_DIR / "Resources_Dataset.csv")

print("CPI shape:", cpi_raw.shape)
print("Resource price shape:", resources_raw.shape)
print("CPI columns:", list(cpi_raw.columns))
print("Resource price columns:", list(resources_raw.columns))

display(cpi_raw.head())
display(resources_raw.head())

CPI shape: (1303, 2)
Resource price shape: (7000, 23)
CPI columns: ['Yearmon', 'CPI']
Resource price columns: ['Date', 'CL=F_Close', 'CL=F_Volume', 'BZ=F_Close', 'BZ=F_Volume', 'GC=F_Close', 'GC=F_Volume', 'SI=F_Close', 'SI=F_Volume', 'NG=F_Close', 'NG=F_Volume', 'ZC=F_Close', 'ZC=F_Volume', 'ZW=F_Close', 'ZW=F_Volume', 'ZS=F_Close', 'ZS=F_Volume', 'HG=F_Close', 'HG=F_Volume', 'PL=F_Close', 'PL=F_Volume', 'PA=F_Close', 'PA=F_Volume']


,Yearmon,CPI
0,01-01-1913,9.8
1,01-02-1913,9.8
2,01-03-1913,9.8
3,01-04-1913,9.8
4,01-05-1913,9.7


,Date,CL=F_Close,CL=F_Volume,BZ=F_Close,BZ=F_Volume,GC=F_Close,GC=F_Volume,SI=F_Close,SI=F_Volume,NG=F_Close,NG=F_Volume,ZC=F_Close,ZC=F_Volume,ZW=F_Close,ZW=F_Volume,ZS=F_Close,ZS=F_Volume,HG=F_Close,HG=F_Volume,PL=F_Close,PL=F_Volume,PA=F_Close,PA=F_Volume
0,29/10/1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,402.700012,1708.0,NaN,NaN
1,30/10/1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,405.299988,1305.0,NaN,NaN
2,31/10/1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,404.200012,862.0,NaN,NaN
3,3/11/1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,406.500000,898.0,NaN,NaN
4,4/11/1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,405.700012,809.0,NaN,NaN


## 2. Clean the CPI data

I changed the date column to `datetime`, kept the CPI values, and calculated monthly and yearly percentage changes.

In [2]:
cpi = cpi_raw.copy()
cpi["Date"] = pd.to_datetime(cpi["Yearmon"], format="%d-%m-%Y")
cpi["CPI"] = pd.to_numeric(cpi["CPI"], errors="coerce")

cpi = cpi[["Date", "CPI"]].sort_values("Date")
cpi = cpi.drop_duplicates(subset="Date")

cpi["Inflation_MoM_pct"] = cpi["CPI"].pct_change(fill_method=None) * 100
cpi["Inflation_YoY_pct"] = cpi["CPI"].pct_change(12, fill_method=None) * 100

print("CPI missing values:")
print(cpi.isna().sum())
display(cpi.head())

cpi.to_csv(OUT_DIR / "cpi_monthly.csv", index=False)

CPI missing values:
Date                  0
CPI                   0
Inflation_MoM_pct     1
Inflation_YoY_pct    12
dtype: int64


,Date,CPI,Inflation_MoM_pct,Inflation_YoY_pct
0,1913-01-01,9.8,NaN,NaN
1,1913-02-01,9.8,0.000000,NaN
2,1913-03-01,9.8,0.000000,NaN
3,1913-04-01,9.8,0.000000,NaN
4,1913-05-01,9.7,-1.020408,NaN


## 3. Clean the resource prices

The resource data is daily. I kept WTI, Brent, and natural gas and calculated the average price for each month.

In [3]:
resources = resources_raw.copy()
resources["Date"] = pd.to_datetime(resources["Date"], format="%d/%m/%Y")

resources = resources[["Date", "CL=F_Close", "BZ=F_Close", "NG=F_Close"]]
resources = resources.rename(columns={
    "CL=F_Close": "WTI",
    "BZ=F_Close": "Brent",
    "NG=F_Close": "Natural_Gas"
})

for col in ["WTI", "Brent", "Natural_Gas"]:
    resources[col] = pd.to_numeric(resources[col], errors="coerce")

monthly = resources.sort_values("Date").set_index("Date")
monthly = monthly.resample("MS").mean().reset_index()

monthly["WTI_Change_pct"] = monthly["WTI"].pct_change(fill_method=None) * 100
monthly["Brent_Change_pct"] = monthly["Brent"].pct_change(fill_method=None) * 100
monthly["Gas_Change_pct"] = monthly["Natural_Gas"].pct_change(fill_method=None) * 100

print("Monthly resource missing values:")
print(monthly.isna().sum())
display(monthly.head())

monthly.to_csv(OUT_DIR / "resource_prices_monthly.csv", index=False)

Monthly resource missing values:
Date                  0
WTI                  34
Brent               117
Natural_Gas          34
WTI_Change_pct       35
Brent_Change_pct    118
Gas_Change_pct       35
dtype: int64


,Date,WTI,Brent,Natural_Gas,WTI_Change_pct,Brent_Change_pct,Gas_Change_pct
0,1997-10-01,NaN,NaN,NaN,NaN,NaN,NaN
1,1997-11-01,NaN,NaN,NaN,NaN,NaN,NaN
2,1997-12-01,NaN,NaN,NaN,NaN,NaN,NaN
3,1998-01-01,NaN,NaN,NaN,NaN,NaN,NaN
4,1998-02-01,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Merge the two datasets

I used `merge()` on the monthly dates, selected the period when the series overlap, and removed rows that are missing values needed for the comparison.

In [4]:
df = pd.merge(cpi, monthly, on="Date", how="inner")

df = df.loc[
    (df["Date"] >= "2008-01-01") & (df["Date"] <= "2021-07-01")
].copy()

needed = [
    "CPI", "Inflation_MoM_pct", "Inflation_YoY_pct",
    "WTI", "Brent", "Natural_Gas",
    "WTI_Change_pct", "Brent_Change_pct", "Gas_Change_pct"
]
df = df.dropna(subset=needed)
df = df.sort_values("Date")

df.to_csv(OUT_DIR / "inflation_resource_monthly.csv", index=False)

print("CPI months:", len(cpi))
print("Resource months:", len(monthly))
print("Merged months:", len(df))
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

df = df.set_index("Date")
display(df.head())

CPI months: 1303
Resource months: 335
Merged months: 163
Date range: 2008-01-01 to 2021-07-01


,CPI,Inflation_MoM_pct,Inflation_YoY_pct,WTI,Brent,Natural_Gas,WTI_Change_pct,Brent_Change_pct,Gas_Change_pct
Date,,,,,,,,,
2008-01-01,211.080,0.497058,4.280294,92.929047,92.112381,7.991476,1.293346,0.786027,11.320502
2008-02-01,211.693,0.290411,4.026555,95.348999,94.652500,8.642250,2.604085,2.757631,8.143349
2008-03-01,213.528,0.866821,3.981456,105.420000,102.798999,9.624300,10.562251,8.606745,11.363359
2008-04-01,214.823,0.606478,3.936890,112.462728,110.441817,10.288091,6.680637,7.434720,6.897031
2008-05-01,216.632,0.842089,4.175543,125.458572,124.380951,11.381619,11.555690,12.621246,10.629067


## 5. Check the cleaned dataset

I checked the merged data before using it for the analysis.

In [5]:
print("Shape:", df.shape)
print("Missing values:")
print(df.isna().sum())
display(df.head())

display(df[["Inflation_MoM_pct", "Inflation_YoY_pct", "WTI", "Brent", "Natural_Gas"]].describe().round(2))

Shape: (163, 9)
Missing values:
CPI                  0
Inflation_MoM_pct    0
Inflation_YoY_pct    0
WTI                  0
Brent                0
Natural_Gas          0
WTI_Change_pct       0
Brent_Change_pct     0
Gas_Change_pct       0
dtype: int64


,CPI,Inflation_MoM_pct,Inflation_YoY_pct,WTI,Brent,Natural_Gas,WTI_Change_pct,Brent_Change_pct,Gas_Change_pct
Date,,,,,,,,,
2008-01-01,211.080,0.497058,4.280294,92.929047,92.112381,7.991476,1.293346,0.786027,11.320502
2008-02-01,211.693,0.290411,4.026555,95.348999,94.652500,8.642250,2.604085,2.757631,8.143349
2008-03-01,213.528,0.866821,3.981456,105.420000,102.798999,9.624300,10.562251,8.606745,11.363359
2008-04-01,214.823,0.606478,3.936890,112.462728,110.441817,10.288091,6.680637,7.434720,6.897031
2008-05-01,216.632,0.842089,4.175543,125.458572,124.380951,11.381619,11.555690,12.621246,10.629067


,Inflation_MoM_pct,Inflation_YoY_pct,WTI,Brent,Natural_Gas
count,163.00,163.00,163.00,163.00,163.00
mean,0.16,1.81,70.86,76.80,3.68
std,0.39,1.36,23.99,26.60,1.78
min,-1.92,-2.10,16.70,26.63,1.70
25%,-0.05,1.14,50.63,55.36,2.70
50%,0.17,1.73,67.85,71.76,3.24
75%,0.42,2.35,93.00,105.57,4.08
max,1.01,5.60,134.02,134.10,12.78


**Output:** The cleaned files are saved in `data/processed/`. The main analysis notebook loads `inflation_resource_monthly.csv`.